Resumo: (tô meio sem tempo)

1) Resolvemos os problemas que encontramos na análise um a um.
2) usamos as formulas 'stdev' e 'median' da biblioteca statistics em algum momentos.
3) usamos a formula 'fillna' para preencher valores vazios, quando necessário.
4) usamos a formula 'loc' para localizar valores problemáticos e substituí-los por valores adequados.

* Colocando todos os dados que precisamos para continuar o trabalho da outra planilha: 

In [5]:
import pandas as pd
import numpy as np
import statistics as stats
import seaborn as seaborn

tabela = pd.read_csv('Churn.csv',sep=';')

tabela.columns = ['Id','Score','Estado','Gênero','Idade','Patrimônio','Saldo','Produtos_Adquiridos','TemCartCred','Ativo','Salário','Saiu']

tabela.head()

,Id,Score,Estado,Gênero,Idade,Patrimônio,Saldo,Produtos_Adquiridos,TemCartCred,Ativo,Salário,Saiu
0,1,619,RS,Feminino,42,2,0,1,1,1,10134888.0,1
1,2,608,SC,Feminino,41,1,8380786,1,0,1,11254258.0,0
2,3,502,RS,Feminino,42,8,1596608,3,1,0,11393157.0,1
3,4,699,RS,Feminino,39,1,0,2,0,0,9382663.0,0
4,5,850,SC,Feminino,43,2,12551082,1,1,1,790841.0,0


* Relembrando o último código para valores nulos (NAN):

In [9]:
tabela.isnull().sum()

Id                     0
Score                  0
Estado                 0
Gênero                 8
Idade                  0
Patrimônio             0
Saldo                  0
Produtos_Adquiridos    0
TemCartCred            0
Ativo                  0
Salário                7
Saiu                   0
dtype: int64

* Tratando a coluna "Salário".

* Os únicos problemas eram os valores vazios e os outliers, então os substituiremos pela mediana (outliers foram feitos lá embaixo): 

In [22]:
mediana_sal = stats.median(tabela['Salário'])
# calculando a mediana usando a biblioteca 'statistics'.

tabela['Salário'].fillna(mediana_sal,inplace = True)
# usando a função 'fillna' (fill non available) para substituir todos os valores
# vazios de tabela['Salário'] para a mediana. 

tabela.isnull().sum()
# checando se os valores vazios foram retirados de tabela['Salário'].  

Id                     0
Score                  0
Estado                 0
Gênero                 8
Idade                  0
Patrimônio             0
Saldo                  0
Produtos_Adquiridos    0
TemCartCred            0
Ativo                  0
Salário                0
Saiu                   0
dtype: int64

* Agora, tratando a coluna "Gênero". 

* Essa coluna tem dois problemas notados anteriormente: Tem valores vazios e tem valores 'Fem', 'F' e 'M' que não estão contidos em "Masculino" e "Feminino".

In [28]:
grupos_genero = tabela.groupby(['Gênero']).size()
grupos_genero

# moda = "Masculino"

Gênero
F              2
Fem            1
Feminino     461
M              6
Masculino    521
dtype: int64

In [31]:
tabela['Gênero'].fillna('Masculino',inplace = True)
tabela.isnull().sum()

# substituindo os valores vazios em tabela['Gênero'] pela moda ("Masculino")

Id                     0
Score                  0
Estado                 0
Gênero                 0
Idade                  0
Patrimônio             0
Saldo                  0
Produtos_Adquiridos    0
TemCartCred            0
Ativo                  0
Salário                0
Saiu                   0
dtype: int64

In [35]:
tabela.loc[tabela['Gênero'] == 'M','Gênero'] = "Masculino"

tabela.loc[tabela['Gênero'] == 'F','Gênero'] = "Feminino"

tabela.loc[tabela['Gênero'] == 'Fem','Gênero'] = "Feminino"

# usando a fórmula 'loc' para localizar todas os valores fora de padrão e 
# atribuindo o valor correto a eles. 

tabela.groupby(['Gênero']).size()

Gênero
Feminino     465
Masculino    535
dtype: int64

* Tratando a coluna "Idade".
* Essa coluna tem o problema de idades negativas e idades muito grandes. Por isso, vamos definir um limite de 0 a 120 anos, e substituir todos os outliers pela mediana.

In [36]:
tabela['Idade'].describe()

count    999.000000
mean      38.902903
std       11.401912
min      -20.000000
25%       32.000000
50%       37.000000
75%       44.000000
max      140.000000
Name: Idade, dtype: float64

In [39]:
mediana_idade = stats.median(tabela['Idade'])
print('Mediana:',mediana_idade)
# definindo a mediana. 

Mediana: 37.0


In [38]:
tabela.loc[tabela['Idade'] < 0, 'Idade'] = mediana_idade
tabela.loc[tabela['Idade'] > 120, 'Idade'] = mediana_idade

# Novamente, usando a função loc para localizar todos os valores que atendem 
# a condição citada e substituindo o seu valor pela mediana.

## Acredito que também poderia ter usado o 'or' e fazer tudo em uma só função.

tabela['Idade'].describe()

count    999.000000
mean      38.903904
std       10.672421
min        0.000000
25%       32.000000
50%       37.000000
75%       44.000000
max       82.000000
Name: Idade, dtype: float64

* Tratando a coluna "Estado".
* O problema era que haviam Estados que não existem ou não pertecem ao Sul. 

In [41]:
grupo_estado = tabela.groupby('Estado').size()
grupo_estado

# moda = "RS"

Estado
PR    257
RP      1
RS    478
SC    258
SP      4
TD      1
dtype: int64

In [42]:
tabela.loc[tabela['Estado'] == 'RP','Estado'] = "RS"
tabela.loc[tabela['Estado'] == 'SP','Estado'] = "RS"
tabela.loc[tabela['Estado'] == 'TD','Estado'] = "RS"

tabela.groupby('Estado').size()

Estado
PR    257
RS    484
SC    258
dtype: int64

* Tratando a coluna "Id".
* Nessa coluna, iremos checar e resolver caso hajam dados duplicados.

In [43]:
tabela[tabela.duplicated(['Id'],keep=False)]

# usaremos essa fórmula 'duplicated()' para encontrar dados duplicados na 
# coluna "Id"

,Id,Score,Estado,Gênero,Idade,Patrimônio,Saldo,Produtos_Adquiridos,TemCartCred,Ativo,Salário,Saiu
80,81.0,665.0,RS,Feminino,34.0,1.0,9664554.0,2.0,0.0,0.0,17141366.0,0.0
81,81.0,665.0,RS,Feminino,34.0,1.0,9664554.0,2.0,0.0,0.0,17141366.0,0.0


In [46]:
tabela.drop_duplicates(subset = 'Id', keep = 'first', inplace = True)

# usando a formula 'drop_duplicates' e requerendo que ela mantenha apenas o 
# primeiro dos dados duplicados, excluímos todos os dados que estejam dessa 
# forma. 

tabela[tabela.duplicated(['Id'],keep = False)]
#checando

,Id,Score,Estado,Gênero,Idade,Patrimônio,Saldo,Produtos_Adquiridos,TemCartCred,Ativo,Salário,Saiu


* Tratando a coluna "Salário".
* O problema dos salários descritos, é que haviam alguns outliers. Valores de salários muito maiores do que todos os outros. Trataremos isso substituindo os valores de todos os salários que estejam muito distantes da média. E usaremos o range de 2 * (desvio padrão) para identificar esses salários.

In [51]:
## IGNORAARRRR
tabela['Salário'] = pd.to_numeric(tabela['Salário'], errors='coerce')
salarios_validos = tabela['Salário'].dropna()
## IGNORAARRRR

dp_salario = stats.stdev(salarios_validos)
dp_salario

# calculando o desvio padrão. 

529254103.7190325

In [53]:
mediana_sal = stats.median(tabela['Salário'])
mediana_sal

# calculando a mediana 

8637195.5

In [54]:
tabela.loc[tabela['Salário'] >= 2 * dp_salario,'Salário']

# identificar algum salário que seja mais que 2 vezes o desvio padrão

7      1.193469e+10
116    1.156383e+10
170    1.640179e+09
230    1.119812e+09
Name: Salário, dtype: float64

In [55]:
tabela.loc[tabela['Salário'] >= 2 * dp_salario,'Salário'] = mediana_sal

# atribuindo o valor da mediana para todos esses valores encontrados. 

In [58]:
tabela.loc[tabela['Salário'] >=  2 * dp_salario] 

# checando se ainda há valores distantes.

,Id,Score,Estado,Gênero,Idade,Patrimônio,Saldo,Produtos_Adquiridos,TemCartCred,Ativo,Salário,Saiu


* Checando nossos novos dados refinados. 

In [59]:
tabela.head()

,Id,Score,Estado,Gênero,Idade,Patrimônio,Saldo,Produtos_Adquiridos,TemCartCred,Ativo,Salário,Saiu
0,1.0,619.0,RS,Feminino,42.0,2.0,0.0,1.0,1.0,1.0,10134888.0,1.0
1,2.0,608.0,SC,Feminino,41.0,1.0,8380786.0,1.0,0.0,1.0,11254258.0,0.0
2,3.0,502.0,RS,Feminino,42.0,8.0,1596608.0,3.0,1.0,0.0,11393157.0,1.0
3,4.0,699.0,RS,Feminino,39.0,1.0,0.0,2.0,0.0,0.0,9382663.0,0.0
4,5.0,850.0,SC,Feminino,43.0,2.0,12551082.0,1.0,1.0,1.0,790841.0,0.0


In [61]:
tabela.shape

(998, 12)